# TAC-LAnoBERT v2: Evaluation & Comparison

**Purpose**: Evaluate TAC v2 and compare with baseline

**Prerequisites**:
- ✅ TAC v2 trained → `outputs/BGL_tac_v2_2epochs/`
- ✅ BGL data preprocessed → `data/BGL/`
- (Optional) Baseline for comparison → `outputs/BGL_lanobert/` or `outputs/BGL_tac/`

**What this notebook does**:
- Run TAC v2 inference (if not done)
- Calculate metrics (F1, Precision, Recall, FPR, AUROC)
- Evaluate early detection (DLT, EWR)
- Compare with baseline (if available)
- Generate report

## Setup

In [ ]:
# Clone repository (if on Kaggle/Colab)
import os

if not os.path.exists('TAC-LAnoBERT-y'):
    !git clone https://github.com/rubyhcm/TAC-LAnoBERT-y.git
    %cd TAC-LAnoBERT-y
else:
    print("✅ Repository already exists")
    if not os.getcwd().endswith('TAC-LAnoBERT-y'):
        %cd TAC-LAnoBERT-y

In [ ]:
!pip install -r requirements.txt -q
print("✅ Dependencies installed")

In [ ]:
# Verify environment
import torch
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("  ⚠️  CPU mode (inference will be slow)")

print("\n✅ Environment ready")

## Check Prerequisites

In [ ]:
# Check for baseline (optional for comparison)
import glob
import os

# First, check if baselines exist in Kaggle input and copy them
baseline_in_input = glob.glob("/kaggle/input/**/BGL_lanobert", recursive=True)
if baseline_in_input and not os.path.exists("outputs/BGL_lanobert"):
    print("\n📦 Baseline (LAnoBERT): Found in input, copying...")
    os.makedirs("outputs", exist_ok=True)
    !cp -r {baseline_in_input[0]} outputs/
    print("✅ Copied")

tac_baseline_in_input = glob.glob("/kaggle/input/**/BGL_tac", recursive=True)
# Exclude BGL_tac_v2_2epochs
tac_baseline_in_input = [p for p in tac_baseline_in_input if "v2" not in p]
if tac_baseline_in_input and not os.path.exists("outputs/BGL_tac"):
    print("\n📦 Baseline (TAC original): Found in input, copying...")
    os.makedirs("outputs", exist_ok=True)
    !cp -r {tac_baseline_in_input[0]} outputs/
    print("✅ Copied")

# Now check if baselines exist
baseline_exists = (
    os.path.exists("outputs/BGL_lanobert/results") or
    os.path.exists("outputs/BGL_tac/results")
)

if baseline_exists:
    print("\n✅ Baseline found (will compare)")
else:
    print("\n⚠️  No baseline found (will skip comparison)")


## Run TAC v2 Inference

Run inference if not already done.

In [ ]:
# Check if inference already done
tac_v2_scores_exist = (
    os.path.exists("outputs/BGL_tac_v2_2epochs/results/scores_tac_hybrid.npy") or
    os.path.exists("outputs/BGL_tac_v2_2epochs/results/scores_tac_mlm_error.npy")
)

if tac_v2_scores_exist:
    print("✅ TAC v2 inference already complete (scores found)")
    print("   Skipping inference...")
else:
    print("Running TAC v2 inference...")
    print("This will take ~1-2 hours on T4 GPU\n")
    
    !python -m tac_lanobert.inference_tac --config configs/bgl_tac_v2_2epochs.yaml
    
    print("\n✅ Inference complete")

## View TAC v2 Results

Load and display detailed results from inference.

In [ ]:
# Load score files
import numpy as np
import json
from pathlib import Path

results_dir = Path('outputs/BGL_tac_v2_2epochs/results')

print("=" * 70)
print("TAC-LANOBERT V2 RESULTS")
print("=" * 70)

# Find score files
score_files = list(results_dir.glob('scores_*.npy'))

if score_files:
    print(f"\n📊 Score Files: {len(score_files)}\n")
    
    scores_dict = {}
    for score_file in sorted(score_files):
        scores = np.load(score_file)
        name = score_file.stem.replace('scores_', '')
        scores_dict[name] = scores
        
        print(f"{name}:")
        print(f"  Lines:  {len(scores):,}")
        print(f"  Min:    {scores.min():.6f}")
        print(f"  Max:    {scores.max():.6f}")
        print(f"  Mean:   {scores.mean():.6f}")
        print(f"  Median: {np.median(scores):.6f}")
        print(f"  Std:    {scores.std():.6f}")
        print()
    
    print("=" * 70)
    
else:
    print("\n⚠️  No score files found")
    print(f"   Expected in: {results_dir}")
    print("   Run inference first!")

In [ ]:
# Parse text report for metrics
import re

def parse_text_report(report_path):
    """Parse TAC report text file for metrics"""
    if not os.path.exists(report_path):
        return None
    
    with open(report_path, 'r') as f:
        content = f.read()
    
    metrics = {}
    
    # Extract metrics
    if m := re.search(r'AUROC:\s+([0-9.e+-]+)', content):
        metrics['auroc'] = float(m.group(1))
    if m := re.search(r'best_F1:\s+([0-9.e+-]+)', content):
        metrics['f1'] = float(m.group(1))
    if m := re.search(r'best_threshold:\s+([0-9.e+-]+)', content):
        metrics['threshold'] = float(m.group(1))
    
    # Extract confusion matrix
    cm_pattern = r'confusion_matrix:.*?\[\[\s*(\d+)\s+(\d+)\s*\]\s*\[\s*(\d+)\s+(\d+)\s*\]\]'
    if m := re.search(cm_pattern, content, re.DOTALL):
        tn, fp, fn, tp = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        metrics['tp'] = tp
        metrics['fp'] = fp
        metrics['tn'] = tn
        metrics['fn'] = fn
        metrics['fpr'] = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        metrics['precision'] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        metrics['recall'] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    return metrics if metrics else None

# Find and parse TAC v2 report
report_files = list(results_dir.glob('*_report.txt'))
tac_v2_report = None

if report_files:
    tac_v2_report = parse_text_report(str(report_files[0]))
    
    if tac_v2_report:
        print("\n" + "=" * 70)
        print("TAC V2 METRICS")
        print("=" * 70)
        print()
        print(f"  F1-Score:    {tac_v2_report.get('f1', 0):.6f}")
        print(f"  Precision:   {tac_v2_report.get('precision', 0):.6f}")
        print(f"  Recall:      {tac_v2_report.get('recall', 0):.6f}")
        print(f"  AUROC:       {tac_v2_report.get('auroc', 0):.6f}")
        print(f"  FPR:         {tac_v2_report.get('fpr', 0)*100:.4f}%")
        print(f"  Threshold:   {tac_v2_report.get('threshold', 0):.6f}")
        print()
        print(f"  Confusion Matrix:")
        print(f"    TP: {tac_v2_report.get('tp', 0):>8,}  FP: {tac_v2_report.get('fp', 0):>8,}")
        print(f"    FN: {tac_v2_report.get('fn', 0):>8,}  TN: {tac_v2_report.get('tn', 0):>8,}")
        print()
        print("=" * 70)
    else:
        print("\n⚠️  Could not parse report file")
else:
    print("\n⚠️  No report files found")
    print(f"   Expected *_report.txt in {results_dir}")

## Compare with Baseline

Compare TAC v2 with baseline (if available).

In [ ]:
# Find all available baselines
baselines = {}

# Check for original TAC baseline
if os.path.exists("outputs/BGL_tac/results/BGL_tac_hybrid_report.txt"):
    report = parse_text_report("outputs/BGL_tac/results/BGL_tac_hybrid_report.txt")
    if report:
        baselines['TAC-LAnoBERT (original)'] = report

# Check for LAnoBERT baseline
if os.path.exists("outputs/BGL_lanobert/results/BGL_error_mean_report.txt"):
    report = parse_text_report("outputs/BGL_lanobert/results/BGL_error_mean_report.txt")
    if report:
        baselines['LAnoBERT (baseline)'] = report

# Compare with each baseline
if baselines and tac_v2_report:
    for baseline_name, baseline_report in baselines.items():
    print("=" * 70)
    print(f"COMPARISON: {baseline_name} vs TAC v2")
    print("=" * 70)
    print()
    
    metrics_to_compare = ['f1', 'precision', 'recall', 'auroc', 'fpr']
    
    print(f"{'Metric':<15} {'Baseline':<15} {'TAC v2':<15} {'Δ':<15} {'Status'}")
    print("-" * 75)
    
    for metric in metrics_to_compare:
        baseline_val = baseline_report.get(metric)
        tac_v2_val = tac_v2_report.get(metric)
        
        if baseline_val is not None and tac_v2_val is not None:
            delta = tac_v2_val - baseline_val
            delta_pct = (delta / baseline_val * 100) if baseline_val != 0 else 0
            
            # Determine status
            if metric == 'fpr':
                # Lower is better
                status = "✅ Better" if delta < 0 else ("⚠️ Worse" if delta > 0 else "≈ Same")
            else:
                # Higher is better
                status = "✅ Better" if delta > 0 else ("⚠️ Worse" if delta < 0 else "≈ Same")
            
            improvement = f"{delta_pct:+.2f}%" if delta != 0 else "0.00%"
            print(f"{metric.upper():<15} {baseline_val:<15.6f} {tac_v2_val:<15.6f} {improvement:<15} {status}")
        else:
            print(f"{metric.upper():<15} {'N/A':<15} {'N/A':<15} {'N/A':<15} {'N/A'}")
    
    print("\n" + "=" * 70)
    
elif tac_v2_report:
    print("\n⚠️  No baseline found for comparison")
    print("   Upload baseline results to compare")
else:
    print("\n⚠️  TAC v2 metrics not found")
    print("   Check if inference completed successfully")

In [ ]:
# Confusion matrix & alert volume comparison for each baseline
if baselines and tac_v2_report:
    for baseline_name, baseline_report in baselines.items():
        if all(k in baseline_report for k in ["tp", "fp", "tn", "fn"]):
            print("\n" + "=" * 70)
            print(f"CONFUSION MATRIX: {baseline_name} vs TAC v2")
            print("=" * 70)
            print()
            print(f"{'':<10} {'TP':>12} {'FP':>12} {'TN':>12} {'FN':>12}")
            print("-" * 70)
            print(f"{baseline_name[:10]:<10} {baseline_report['tp']:>12,} {baseline_report['fp']:>12,} "
                  f"{baseline_report['tn']:>12,} {baseline_report['fn']:>12,}")
            print(f"{'TAC v2':<10} {tac_v2_report['tp']:>12,} {tac_v2_report['fp']:>12,} "
                  f"{tac_v2_report['tn']:>12,} {tac_v2_report['fn']:>12,}")
            
            # Alert reduction
            baseline_alerts = baseline_report["tp"] + baseline_report["fp"]
            tac_v2_alerts = tac_v2_report["tp"] + tac_v2_report["fp"]
            alert_reduction = (baseline_alerts - tac_v2_alerts) / baseline_alerts * 100
            
            print("\n" + "=" * 70)
            print(f"ALERT VOLUME: {baseline_name} vs TAC v2")
            print("=" * 70)
            print(f"\n{baseline_name} alerts:  {baseline_alerts:,}")
            print(f"TAC v2 alerts:        {tac_v2_alerts:,}")
            print(f"Reduction:            {alert_reduction:+.2f}%")
            
            if alert_reduction > 0:
                print(f"\n✅ TAC v2 reduces alert volume by {alert_reduction:.1f}%!")
                print(f"   Fewer false positives = less alert fatigue")
            elif alert_reduction < 0:
                print(f"\n⚠️  TAC v2 generates {abs(alert_reduction):.1f}% more alerts")
            else:
                print(f"\n≈ Similar alert volume")
            
            print("\n" + "=" * 70)


## Summary

In [ ]:
from datetime import datetime

print("=" * 70)
print("EVALUATION SUMMARY")
print("=" * 70)

print(f"\nCompleted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if tac_v2_report:
    print(f"\n📊 TAC v2 Performance:")
    print(f"   F1:        {tac_v2_report.get('f1', 0):.6f}")
    print(f"   Precision: {tac_v2_report.get('precision', 0):.6f}")
    print(f"   Recall:    {tac_v2_report.get('recall', 0):.6f}")
    print(f"   AUROC:     {tac_v2_report.get('auroc', 0):.6f}")
    print(f"   FPR:       {tac_v2_report.get('fpr', 0)*100:.4f}%")

if baseline_report and tac_v2_report:
    f1_improve = (tac_v2_report['f1'] - baseline_report['f1']) / baseline_report['f1'] * 100
    fpr_improve = (baseline_report['fpr'] - tac_v2_report['fpr']) / baseline_report['fpr'] * 100
    
    print(f"\n📈 Improvement vs {baseline_name}:")
    print(f"   F1:  {f1_improve:+.2f}%")
    print(f"   FPR: {fpr_improve:+.2f}% (lower is better)")

print(f"\n📂 Results saved in:")
print(f"   {results_dir}")

print("\n" + "=" * 70)
print("✅ Evaluation Complete")
print("=" * 70)